In [1]:
# 1. Import Library
import pandas as pd
import yaml
import os
import sys

from pathlib import Path

In [2]:
# 2. Import Function
sys.path.append(os.path.abspath("../.."))

from src.data_preparation import (
    get_data_predict,
    remove_first_year_per_ticker,
    remove_duplicates,
    remove_null_rows,
    split_train_val_test,
    remove_same_ohlc_rows,
    remove_zero_volume_rows,
    drop_columns
)

In [3]:
# 3. Load Parameter
with open("../../config/parameter.yaml", "r") as f:
    parameter_config = yaml.safe_load(f)

data_parameter = parameter_config["data_parameter"]
columns_parameter = parameter_config["columns_parameter"]

start_train = data_parameter["start_train"]
end_train = data_parameter["end_train"]
start_val = data_parameter["start_val"]
end_val = data_parameter["end_val"]
start_test = data_parameter["start_test"]
end_test = data_parameter["end_test"]
predict_date = data_parameter["predict_date"]

feature_to_drop = columns_parameter["feature_to_drop"]
label_to_drop = columns_parameter["label_to_drop"]
label_to_drop_simulation = columns_parameter["label_to_drop_simulation"]
label_to_drop_predict = columns_parameter["label_to_drop_predict"]

print(f"Periode data train      : {start_train} to {end_train}")
print(f"Periode data validation : {start_val} to {end_val}")
print(f"Periode data test       : {start_test} to {end_test}")
print(f"Predict date            : {predict_date}")

print(f"feature_to_drop          : {feature_to_drop}")
print(f"label_to_drop            : {label_to_drop}")
print(f"label_to_drop_simulation : {label_to_drop_simulation}")
print(f"label_to_drop_predict    : {label_to_drop_predict}")

Periode data train      : 2020-01-01 to 2024-01-01
Periode data validation : 2024-01-01 to 2025-01-01
Periode data test       : 2025-01-01 to 2026-01-01
Predict date            : 2026-01-14
feature_to_drop          : ['volume_sma_21', 'vwap_21', 'ema_9', 'ema_21', 'ema_50', 'ema_200', 'rsi_14', 'rsi_sma_14', 'macd_12_26', 'macd_signal_ema_9', 'macd_histogram', 'atr_14', 'bb_mid_20', 'bb_upper_20', 'bb_lower_20', 'highest_50', 'highest_200', 'garman_klass_variance']
label_to_drop            : ['next_close', 'next_close_pct', 'rank', 'rank_class']
label_to_drop_simulation : ['next_close', 'rank', 'rank_class']
label_to_drop_predict    : ['rank_pct']


In [4]:
# 4. Load Data
with open("../../config/paths.yaml", "r") as f:
    paths_config = yaml.safe_load(f)

labels_data_path = paths_config["data"]["03_labels"]
raw_data_path = paths_config["data"]["01_raw"]

csv_file_path = os.path.join(labels_data_path, "labeled_technicals.csv")
csv_file_path_index = os.path.join(raw_data_path, "raw_technicals_index.csv")

df_all_data = pd.read_csv(csv_file_path, parse_dates=["date"])
df_all_data_index = pd.read_csv(csv_file_path_index, parse_dates=["date"])

In [5]:
# 5. Data Preparation
df_data_predict = get_data_predict(df_all_data, predict_date=predict_date)

df_all_data = remove_first_year_per_ticker(df_all_data)
df_all_data = remove_duplicates(df_all_data)
df_all_data = remove_null_rows(df_all_data)
df_all_data_index = remove_first_year_per_ticker(df_all_data_index)
df_all_data_index = remove_duplicates(df_all_data_index)
df_all_data_index = remove_null_rows(df_all_data_index)

df_data_train, df_data_val, df_data_test = split_train_val_test(
    df_all_data,
    start_train=start_train,
    end_train=end_train,
    start_val=start_val,
    end_val=end_val,
    start_test=start_test,
    end_test=end_test
)

df_data_train_index, df_data_val_index, df_data_test_index = split_train_val_test(
    df_all_data_index,
    start_train=start_train,
    end_train=end_train,
    start_val=start_val,
    end_val=end_val,
    start_test=start_test,
    end_test=end_test
)

df_data_train_index = remove_same_ohlc_rows(df_data_train_index)
df_data_train_index = remove_zero_volume_rows(df_data_train_index)

columns_to_drop = feature_to_drop + label_to_drop
columns_to_drop_simulation = feature_to_drop + label_to_drop_simulation
columns_to_drop_predict = feature_to_drop + label_to_drop + label_to_drop_predict

df_data_simulation = drop_columns(df_data_test, columns_to_drop_simulation)
df_data_train = drop_columns(df_data_train, columns_to_drop)
df_data_val = drop_columns(df_data_val, columns_to_drop)
df_data_test = drop_columns(df_data_test, columns_to_drop)
df_data_predict = drop_columns(df_data_predict, columns_to_drop_predict)

In [6]:
# 6. Export to CSV
with open("../../config/paths.yaml", "r") as f:
    paths_config = yaml.safe_load(f)

preparation_data_path = Path(paths_config["data"]["05_data_preparation"])
preparation_data_path.mkdir(parents=True, exist_ok=True)

output_file_train = preparation_data_path / "01_train_prep_technicals.csv"
output_file_val = preparation_data_path / "02_val_prep_technicals.csv"
output_file_test = preparation_data_path / "03_test_prep_technicals.csv"
output_file_simulation = preparation_data_path / "simulation_data.csv"
output_file_predict = preparation_data_path / "predict_data.csv"
output_file_train_index = preparation_data_path / "01_train_prep_technicals_index.csv"
output_file_val_index = preparation_data_path / "02_val_prep_technicals_index.csv"
output_file_test_index = preparation_data_path / "03_test_prep_technicals_index.csv"

df_data_train.to_csv(output_file_train, index=False)
df_data_val.to_csv(output_file_val, index=False)
df_data_test.to_csv(output_file_test, index=False)
df_data_simulation.to_csv(output_file_simulation, index=False)
df_data_predict.to_csv(output_file_predict, index=False)
df_data_train_index.to_csv(output_file_train_index, index=False)
df_data_val_index.to_csv(output_file_val_index, index=False)
df_data_test_index.to_csv(output_file_test_index, index=False)

print(f"Data train berhasil disimpan ke            : '{output_file_train}'")
print(f"Data validation berhasil disimpan ke       : '{output_file_val}'")
print(f"Data test berhasil disimpan ke             : '{output_file_test}'")
print(f"Data simulation berhasil disimpan ke       : '{output_file_simulation}'")
print(f"Data predict berhasil disimpan ke          : '{output_file_predict}'")
print(f"Data train index berhasil disimpan ke      : '{output_file_train_index}'")
print(f"Data validation index berhasil disimpan ke : '{output_file_val_index}'")
print(f"Data test index berhasil disimpan ke       : '{output_file_test_index}'")

print(f"Total baris data train            : {len(df_data_train):,}")
print(f"Total baris data validation       : {len(df_data_val):,}")
print(f"Total baris data test             : {len(df_data_test):,}")
print(f"Total baris data simulation       : {len(df_data_simulation):,}")
print(f"Total baris data predict          : {len(df_data_predict):,}")
print(f"Total baris data train index      : {len(df_data_train_index):,}")
print(f"Total baris data validation index : {len(df_data_val_index):,}")
print(f"Total baris data test index       : {len(df_data_test_index):,}")

Data train berhasil disimpan ke            : '..\..\data\05_data_preparation\01_train_prep_technicals.csv'
Data validation berhasil disimpan ke       : '..\..\data\05_data_preparation\02_val_prep_technicals.csv'
Data test berhasil disimpan ke             : '..\..\data\05_data_preparation\03_test_prep_technicals.csv'
Data simulation berhasil disimpan ke       : '..\..\data\05_data_preparation\simulation_data.csv'
Data predict berhasil disimpan ke          : '..\..\data\05_data_preparation\predict_data.csv'
Data train index berhasil disimpan ke      : '..\..\data\05_data_preparation\01_train_prep_technicals_index.csv'
Data validation index berhasil disimpan ke : '..\..\data\05_data_preparation\02_val_prep_technicals_index.csv'
Data test index berhasil disimpan ke       : '..\..\data\05_data_preparation\03_test_prep_technicals_index.csv'
Total baris data train            : 67,853
Total baris data validation       : 18,420
Total baris data test             : 18,657
Total baris data simulat

In [7]:
df_data_train.head(9999999)

,ticker,date,open,high,low,close,adj_close,volume,cmf_5,cmf_5_delta,...,z_score_return_21,month_sin,month_cos,regime,rsi_14_regime,rsi_sma_14_regime,hidden_demand,hidden_supply,money_flow_magnitude,rank_pct
0,ACES.JK,2020-01-02,1500.0,1535.0,1500.0,1510.0,1217.50,13727900.0,-0.533880,-0.377952,...,0.530427,5.000000e-01,0.866025,1.0,1.0,1.0,0.0,1.0,0.293183,0.4706
1,ACES.JK,2020-01-03,1520.0,1525.0,1500.0,1515.0,1221.53,15248800.0,-0.308721,0.225159,...,0.210870,5.000000e-01,0.866025,1.0,1.0,1.0,0.0,0.0,0.148729,0.1045
2,ACES.JK,2020-01-06,1515.0,1515.0,1470.0,1470.0,1185.24,18050000.0,-0.501549,-0.192828,...,-1.105755,5.000000e-01,0.866025,1.0,1.0,1.0,0.0,0.0,0.182503,0.1343
3,ACES.JK,2020-01-07,1470.0,1485.0,1415.0,1440.0,1161.06,34751300.0,-0.478246,0.023303,...,-0.733658,5.000000e-01,0.866025,0.0,1.0,1.0,1.0,0.0,-0.129177,0.1791
4,ACES.JK,2020-01-08,1440.0,1440.0,1395.0,1410.0,1136.87,32211300.0,-0.364505,0.113741,...,-0.701454,5.000000e-01,0.866025,0.0,1.0,1.0,1.0,0.0,-0.055039,0.9118
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67848,UNVR.JK,2023-12-21,3470.0,3480.0,3440.0,3470.0,3103.09,4497500.0,0.380814,0.160857,...,0.210496,-2.449294e-16,1.000000,1.0,1.0,1.0,0.0,0.0,-0.246232,0.4620
67849,UNVR.JK,2023-12-22,3470.0,3490.0,3440.0,3470.0,3103.09,6665700.0,-0.012085,-0.392899,...,0.056606,-2.449294e-16,1.000000,1.0,1.0,1.0,0.0,0.0,0.005509,0.4810
67850,UNVR.JK,2023-12-27,3470.0,3520.0,3450.0,3470.0,3103.09,7000200.0,0.018623,0.030708,...,0.047608,-2.449294e-16,1.000000,1.0,1.0,1.0,0.0,0.0,-0.008090,0.8228
67851,UNVR.JK,2023-12-28,3500.0,3570.0,3500.0,3540.0,3165.68,12314500.0,-0.017173,-0.035796,...,1.270209,-2.449294e-16,1.000000,1.0,2.0,1.0,0.0,1.0,0.000111,0.3671


In [8]:
df_data_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67853 entries, 0 to 67852
Data columns (total 50 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   ticker                      67853 non-null  object        
 1   date                        67853 non-null  datetime64[ns]
 2   open                        67853 non-null  float64       
 3   high                        67853 non-null  float64       
 4   low                         67853 non-null  float64       
 5   close                       67853 non-null  float64       
 6   adj_close                   67853 non-null  float64       
 7   volume                      67853 non-null  float64       
 8   cmf_5                       67853 non-null  float64       
 9   cmf_5_delta                 67853 non-null  float64       
 10  mfi_14                      67853 non-null  float64       
 11  price_level                 67853 non-null  int64     

In [9]:
df_data_predict.head(9999999)

,ticker,date,open,high,low,close,adj_close,volume,cmf_5,cmf_5_delta,...,rolling_std_return_21,z_score_return_21,month_sin,month_cos,regime,rsi_14_regime,rsi_sma_14_regime,hidden_demand,hidden_supply,money_flow_magnitude
0,AADI.JK,2026-01-14,7550.0,7700.0,7450.0,7450.0,7450.0,18886700.0,-0.234402,-0.398549,...,0.017737,-0.035578,0.5,0.866025,3.0,2.0,1.0,0.0,0.0,-0.202449
1,ACES.JK,2026-01-14,410.0,412.0,408.0,410.0,410.0,33489800.0,-0.351181,0.247619,...,0.007442,0.120062,0.5,0.866025,0.0,1.0,1.0,0.0,0.0,-0.068755
2,ADMR.JK,2026-01-14,2030.0,2070.0,1960.0,1975.0,1975.0,83825700.0,-0.014574,-0.086322,...,0.053402,-0.656549,0.5,0.866025,2.0,3.0,2.0,0.0,0.0,0.002561
3,ADRO.JK,2026-01-14,2300.0,2330.0,2210.0,2230.0,2230.0,225784000.0,-0.085044,-0.170077,...,0.032075,-1.126911,0.5,0.866025,3.0,3.0,2.0,0.0,0.0,-0.020116
4,AKRA.JK,2026-01-14,1245.0,1270.0,1235.0,1245.0,1245.0,27776600.0,-0.296689,0.016509,...,0.010882,0.452291,0.5,0.866025,0.0,1.0,2.0,0.0,0.0,-0.218475
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,TKIM.JK,2026-01-14,8150.0,8250.0,8050.0,8100.0,8100.0,1539200.0,-0.201647,0.080159,...,0.018354,-0.546872,0.5,0.866025,3.0,3.0,2.0,1.0,0.0,-0.000812
76,TLKM.JK,2026-01-14,3690.0,3700.0,3610.0,3650.0,3650.0,98179400.0,-0.076640,-0.104632,...,0.015849,0.507134,0.5,0.866025,3.0,2.0,2.0,0.0,1.0,-0.008481
77,TOWR.JK,2026-01-14,555.0,560.0,550.0,550.0,550.0,42360600.0,-0.506056,-0.160321,...,0.021859,-0.442393,0.5,0.866025,1.0,1.0,2.0,0.0,0.0,0.008655
78,UNTR.JK,2026-01-14,31450.0,31825.0,31125.0,31500.0,31500.0,2675500.0,0.240846,0.066492,...,0.014692,-0.057058,0.5,0.866025,2.0,2.0,2.0,0.0,0.0,-0.086536


In [10]:
df_data_predict.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 49 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   ticker                      80 non-null     object        
 1   date                        80 non-null     datetime64[ns]
 2   open                        80 non-null     float64       
 3   high                        80 non-null     float64       
 4   low                         80 non-null     float64       
 5   close                       80 non-null     float64       
 6   adj_close                   80 non-null     float64       
 7   volume                      80 non-null     float64       
 8   cmf_5                       80 non-null     float64       
 9   cmf_5_delta                 80 non-null     float64       
 10  mfi_14                      80 non-null     float64       
 11  price_level                 80 non-null     int64         
 